In [ ]:
import pandas as pd
import numpy as np
import re


# Pauling electronegativities
PAULING_EN = {
    'H': 2.20, 'He': 0.00,
    'Li': 0.98, 'Be': 1.57, 'B': 2.04, 'C': 2.55, 'N': 3.04, 'O': 3.44, 'F': 3.98,
    'Na': 0.93, 'Mg': 1.31, 'Al': 1.61, 'Si': 1.90, 'P': 2.19, 'S': 2.58, 'Cl': 3.16,
    'K': 0.82, 'Ca': 1.00, 'Sc': 1.36, 'Ti': 1.54, 'V': 1.63, 'Cr': 1.66, 
    'Mn': 1.55, 'Fe': 1.83, 'Co': 1.88, 'Ni': 1.91, 'Cu': 1.90, 'Zn': 1.65,
    'Ga': 1.81, 'Ge': 2.01, 'As': 2.18, 'Se': 2.55, 'Br': 2.96,
    'Rb': 0.82, 'Sr': 0.95, 'Y': 1.22, 'Zr': 1.33, 'Nb': 1.6, 'Mo': 2.16,
    'Tc': 1.9, 'Ru': 2.2, 'Rh': 2.28, 'Pd': 2.20, 'Ag': 1.93, 'Cd': 1.69,
    'In': 1.78, 'Sn': 1.96, 'Sb': 2.05, 'Te': 2.1, 'I': 2.66,
    'Cs': 0.79, 'Ba': 0.89, 'La': 1.10, 'Ce': 1.12, 'Pr': 1.13, 'Nd': 1.14,
    'Pm': 1.13, 'Sm': 1.17, 'Eu': 1.2, 'Gd': 1.20, 'Tb': 1.1, 'Dy': 1.22,
    'Ho': 1.23, 'Er': 1.24, 'Tm': 1.25, 'Yb': 1.1, 'Lu': 1.27,
    'Hf': 1.3, 'Ta': 1.5, 'W': 2.36, 'Re': 1.9, 'Os': 2.2, 'Ir': 2.20,
    'Pt': 2.28, 'Au': 2.54, 'Hg': 2.00, 'Tl': 1.62, 'Pb': 2.33, 'Bi': 2.02,
    'Po': 2.0, 'At': 2.2, 'Rn': 0.00
}

def parse_formula(formula):
    """Parse chemical formula like Ta4Bi8O22 and return element counts"""
    # Match element symbol (capital + optional lowercase) followed by optional number
    pattern = r'([A-Z][a-z]?)(\d*)'
    matches = re.findall(pattern, formula)
    
    element_counts = {}
    for element, count in matches:
        if element:  # Skip empty matches
            count = int(count) if count else 1
            element_counts[element] = element_counts.get(element, 0) + count
    
    return element_counts

def calculate_en_features(formula):
    elements = parse_formula(formula)
    
    if not elements:
        return pd.Series({
            'en_mean': np.nan,
            'en_max': np.nan,
            'en_min': np.nan,
            'en_range': np.nan,
            'en_std': np.nan,
            'en_weighted_mean': np.nan
        })
    
    # Get electronegativities and counts
    en_values = []
    counts = []
    
    for element, count in elements.items():
        if element in PAULING_EN:
            en_values.append(PAULING_EN[element])
            counts.append(count)
    
    if not en_values:
        return pd.Series({
            'en_mean': np.nan,
            'en_max': np.nan,
            'en_min': np.nan,
            'en_range': np.nan,
            'en_std': np.nan,
            'en_weighted_mean': np.nan
        })
    
    return pd.Series({
        'en_mean': np.mean(en_values),
        'en_max': np.max(en_values),
        'en_min': np.min(en_values),
        'en_range': np.max(en_values) - np.min(en_values),
        'en_std': np.std(en_values) if len(en_values) > 1 else 0,
        'en_weighted_mean': np.average(en_values, weights=counts)
    })

# Test with your example
test_formula = 'Ta4Bi8O22'
print(f"Formula: {test_formula}")
print(f"Parsed: {parse_formula(test_formula)}")
print(f"\nFeatures:")
print(calculate_en_features(test_formula))

h_featurized = pd.concat([h_featurized, h_featurized['bulk_symbols'].apply(calculate_en_features)], axis=1)
no_ads_featurized = pd.concat([no_ads_featurized, no_ads_featurized['bulk_symbols'].apply(calculate_en_features)], axis=1)

Formula: Ta4Bi8O22
Parsed: {'Ta': 4, 'Bi': 8, 'O': 22}

Features:
en_mean             2.320000
en_max              3.440000
en_min              1.500000
en_range            1.940000
en_std              0.819919
en_weighted_mean    2.877647
dtype: float64


In [ ]:
import pandas as pd
import numpy as np
import re
from itertools import product

# Pauling electronegativities
PAULING_EN = {
    'H': 2.20, 'He': 0.00,
    'Li': 0.98, 'Be': 1.57, 'B': 2.04, 'C': 2.55, 'N': 3.04, 'O': 3.44, 'F': 3.98,
    'Na': 0.93, 'Mg': 1.31, 'Al': 1.61, 'Si': 1.90, 'P': 2.19, 'S': 2.58, 'Cl': 3.16,
    'K': 0.82, 'Ca': 1.00, 'Sc': 1.36, 'Ti': 1.54, 'V': 1.63, 'Cr': 1.66, 
    'Mn': 1.55, 'Fe': 1.83, 'Co': 1.88, 'Ni': 1.91, 'Cu': 1.90, 'Zn': 1.65,
    'Ga': 1.81, 'Ge': 2.01, 'As': 2.18, 'Se': 2.55, 'Br': 2.96,
    'Rb': 0.82, 'Sr': 0.95, 'Y': 1.22, 'Zr': 1.33, 'Nb': 1.6, 'Mo': 2.16,
    'Tc': 1.9, 'Ru': 2.2, 'Rh': 2.28, 'Pd': 2.20, 'Ag': 1.93, 'Cd': 1.69,
    'In': 1.78, 'Sn': 1.96, 'Sb': 2.05, 'Te': 2.1, 'I': 2.66,
    'Cs': 0.79, 'Ba': 0.89, 'La': 1.10, 'Ce': 1.12, 'Pr': 1.13, 'Nd': 1.14,
    'Pm': 1.13, 'Sm': 1.17, 'Eu': 1.2, 'Gd': 1.20, 'Tb': 1.1, 'Dy': 1.22,
    'Ho': 1.23, 'Er': 1.24, 'Tm': 1.25, 'Yb': 1.1, 'Lu': 1.27,
    'Hf': 1.3, 'Ta': 1.5, 'W': 2.36, 'Re': 1.9, 'Os': 2.2, 'Ir': 2.20,
    'Pt': 2.28, 'Au': 2.54, 'Hg': 2.00, 'Tl': 1.62, 'Pb': 2.33, 'Bi': 2.02,
    'Po': 2.0, 'At': 2.2, 'Rn': 0.00
}

# Define metals vs nonmetals (simple classification)
# Nonmetals (including metalloids for simplicity)
NONMETALS = {'H', 'C', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Se', 'Br', 'I', 'At',
             'He', 'Ne', 'Ar', 'Kr', 'Xe', 'Rn',  # Noble gases
             'B', 'Si', 'Ge', 'As', 'Sb', 'Te', 'Po'}  # Metalloids

def is_metal(element):
    """Check if element is a metal"""
    return element not in NONMETALS

def parse_formula(formula):
    """Parse chemical formula and return element counts"""
    pattern = r'([A-Z][a-z]?)(\d*)'
    matches = re.findall(pattern, formula)
    
    element_counts = {}
    for element, count in matches:
        if element:
            count = int(count) if count else 1
            element_counts[element] = element_counts.get(element, 0) + count
    
    return element_counts

def calculate_metal_nonmetal_en_diff(formula):
    """
    Calculate electronegativity differences between metals and nonmetals
    assuming equal coordination
    """
    elements = parse_formula(formula)
    
    if not elements:
        return pd.Series({
            'en_diff_mean': np.nan,
            'en_diff_max': np.nan,
            'en_diff_min': np.nan,
            'en_diff_weighted_mean': np.nan,
            'en_diff_std': np.nan
        })
    
    # Separate metals and nonmetals
    metals = {}
    nonmetals = {}
    
    for element, count in elements.items():
        if element in PAULING_EN:
            if is_metal(element):
                metals[element] = {'count': count, 'en': PAULING_EN[element]}
            else:
                nonmetals[element] = {'count': count, 'en': PAULING_EN[element]}
    
    # If no metals or no nonmetals, can't calculate differences
    if not metals or not nonmetals:
        return pd.Series({
            'en_diff_mean': np.nan,
            'en_diff_max': np.nan,
            'en_diff_min': np.nan,
            'en_diff_weighted_mean': np.nan,
            'en_diff_std': np.nan
        })
    
    # Calculate all metal-nonmetal pairs
    en_diffs = []
    weights = []
    
    for metal, m_data in metals.items():
        for nonmetal, nm_data in nonmetals.items():
            # Electronegativity difference (absolute value)
            en_diff = abs(nm_data['en'] - m_data['en'])
            en_diffs.append(en_diff)
            
            # Weight by the product of their counts (coordination assumption)
            weight = m_data['count'] * nm_data['count']
            weights.append(weight)
    
    if not en_diffs:
        return pd.Series({
            'en_diff_mean': np.nan,
            'en_diff_max': np.nan,
            'en_diff_min': np.nan,
            'en_diff_weighted_mean': np.nan,
            'en_diff_std': np.nan
        })
    
    return pd.Series({
        'en_diff_mean': np.mean(en_diffs),
        'en_diff_max': np.max(en_diffs),
        'en_diff_min': np.min(en_diffs),
        'en_diff_weighted_mean': np.average(en_diffs, weights=weights),
        'en_diff_std': np.std(en_diffs) if len(en_diffs) > 1 else 0
    })

# Test with your example
test_formula = 'Ta4Bi8O22'
print(f"Formula: {test_formula}")
elements = parse_formula(test_formula)
print(f"Parsed: {elements}")

# Show classification
metals = {el: count for el, count in elements.items() if is_metal(el)}
nonmetals = {el: count for el, count in elements.items() if not is_metal(el)}
print(f"\nMetals: {metals}")
print(f"Nonmetals: {nonmetals}")

print(f"\nElectronegativity Difference Features:")
print(calculate_metal_nonmetal_en_diff(test_formula))

h_featurized = pd.concat([h_featurized, h_featurized['bulk_symbols'].apply(calculate_metal_nonmetal_en_diff)], axis=1)
no_ads_featurized = pd.concat([no_ads_featurized, no_ads_featurized['bulk_symbols'].apply(calculate_metal_nonmetal_en_diff)], axis=1)

Formula: Ta4Bi8O22
Parsed: {'Ta': 4, 'Bi': 8, 'O': 22}

Metals: {'Ta': 4, 'Bi': 8}
Nonmetals: {'O': 22}

Electronegativity Difference Features:
en_diff_mean             1.680000
en_diff_max              1.940000
en_diff_min              1.420000
en_diff_weighted_mean    1.593333
en_diff_std              0.260000
dtype: float64
